# Tugas Besar 2 IF3270 — CNN Image Classification
Dataset: Intel Image Classification (~25.000 gambar, 6 kelas)

## 0. Setup Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT      = '/content/drive/MyDrive'
ZIP_PATH        = '/content/drive/MyDrive/Tubes2ML/PixelToCaption.zip'
INTEL_DATA_PATH = '/content/drive/MyDrive/Tubes2ML/intel'

print(f'ZIP ada   : {os.path.exists(ZIP_PATH)}')
print(f'Intel ada : {os.path.exists(INTEL_DATA_PATH)}')

In [ ]:
import zipfile

REPO_DIR = '/content/PixelToCaption'

if not os.path.exists(REPO_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print('Ekstraksi selesai')
else:
    print('Repo sudah ada')

print(os.listdir(REPO_DIR))

In [ ]:
open(os.path.join(REPO_DIR, 'src', '__init__.py'), 'a').close()
open(os.path.join(REPO_DIR, 'src', 'cnn', '__init__.py'), 'a').close()
print('__init__.py dibuat')

In [ ]:
!pip install scikit-learn tqdm -q

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

DATA_DIR    = INTEL_DATA_PATH
WEIGHTS_DIR = os.path.join(DRIVE_ROOT, 'weights', 'cnn')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'DATA_DIR    : {DATA_DIR}')
print(f'WEIGHTS_DIR : {WEIGHTS_DIR}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, json
import tensorflow as tf
from tensorflow import keras

from src.cnn.train import (
    load_dataset, build_cnn_model, build_locally_connected_model,
    run_experiments, CLASS_NAMES, IMG_SIZE, EPOCHS
)
from src.cnn.cnn import CNNScratch
from src.cnn.layers import (
    Conv2DLayer, MaxPooling2DLayer, AveragePooling2DLayer,
    GlobalAveragePooling2DLayer, FlattenLayer,
    LocallyConnected2DLayer, ActivationLayer
)
from src.shared.dense import DenseLayer
from src.shared.image_utils import load_image_cnn, load_batch_cnn, extract_features_cnn
from src.cnn.utils import (
    evaluate_keras_model, evaluate_scratch_model,
    plot_history, plot_f1_comparison
)

import src.cnn.train as train_module
train_module.WEIGHTS_DIR = WEIGHTS_DIR

print(f'GPU : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Kelas: {CLASS_NAMES}')

## 1. Load Dataset

In [ ]:
train_ds, val_ds = load_dataset(DATA_DIR)

X_test, y_test = [], []
for X_batch, y_batch in val_ds:
    X_test.append(X_batch.numpy())
    y_test.append(y_batch.numpy())
X_test = np.concatenate(X_test)
y_test = np.concatenate(y_test)

print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for cls_idx, ax in enumerate(axes):
    ax.imshow(X_test[y_test == cls_idx][0])
    ax.set_title(CLASS_NAMES[cls_idx])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Definisi Konfigurasi

In [ ]:
filter_variants = {
    'small': {2: [32, 64],    3: [32, 64, 64]},
    'large': {2: [64, 128],   3: [64, 128, 128]}
}
kernel_variants = {
    'k3': {2: [3, 3],   3: [3, 3, 3]},
    'k5': {2: [5, 5],   3: [5, 5, 5]}
}

all_configs = []
for n_layers in [2, 3]:
    for fkey, f_dict in filter_variants.items():
        for kkey, k_dict in kernel_variants.items():
            for pool in ['max', 'average']:
                all_configs.append({
                    'name':            f'conv{n_layers}_{fkey}_{kkey}_{pool}',
                    'num_conv_layers': n_layers,
                    'filters':         f_dict[n_layers],
                    'kernel_sizes':    k_dict[n_layers],
                    'pooling_type':    pool
                })

print(f'Total konfigurasi: {len(all_configs)}')

## 3. Training Semua Variasi Conv2D (16 Arsitektur)
> **SKIP cell ini dan jalankan cell berikutnya jika bobot sudah ada di Drive**

In [ ]:
existing = [f for f in os.listdir(WEIGHTS_DIR) if f.endswith('.weights.h5')]
print(f'Bobot yang sudah ada ({len(existing)}): {existing}')

In [ ]:
# jalankan training — bobot dan history disimpan ke Drive
results = run_experiments(DATA_DIR)

for name, res in results.items():
    h_path = os.path.join(WEIGHTS_DIR, f'{name}_history.json')
    with open(h_path, 'w') as f:
        json.dump(res['history'], f)

print('Training selesai, history disimpan ke Drive')

In [ ]:
# load history dari Drive — jalankan ini jika cell training di-skip
results = {}
for cfg in all_configs:
    name   = cfg['name']
    h_path = os.path.join(WEIGHTS_DIR, f'{name}_history.json')
    if os.path.exists(h_path):
        with open(h_path) as f:
            results[name] = {'history': json.load(f), 'config': cfg}

print(f'History loaded: {len(results)} model')

## 4. Evaluasi Semua Variasi — Macro F1-Score

In [ ]:
all_results = {}

for cfg in all_configs:
    w_path = os.path.join(WEIGHTS_DIR, f"{cfg['name']}.weights.h5")
    if not os.path.exists(w_path):
        print(f"SKIP {cfg['name']}")
        continue

    model = build_cnn_model(
        cfg['num_conv_layers'], cfg['filters'],
        cfg['kernel_sizes'],    cfg['pooling_type']
    )
    model.load_weights(w_path)
    res = evaluate_keras_model(model, X_test, y_test, CLASS_NAMES)
    all_results[cfg['name']] = {**res, 'config': cfg}
    print(f"{cfg['name']:<35} F1 = {res['macro_f1']:.4f}")

print(f'\nTotal: {len(all_results)}')

## 5. Analisis Pengaruh Jumlah Conv Layer

In [ ]:
labels, scores = [], []
for n in [2, 3]:
    name = f'conv{n}_large_k3_max'
    labels.append(f'{n} layers')
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jumlah Conv Layer')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for n in [2, 3]:
    name = f'conv{n}_large_k3_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=f'{n} layers')
    axes[1].plot(h['val_loss'], label=f'{n} layers')
axes[0].set_title('Train Loss'); axes[1].set_title('Val Loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jumlah Conv Layer')
plt.tight_layout(); plt.show()

## 6. Analisis Pengaruh Jumlah Filter

In [ ]:
filter_labels = {'small': '[32,64]', 'large': '[64,128]'}
labels, scores = [], []
for fkey in ['small', 'large']:
    name = f'conv2_{fkey}_k3_max'
    labels.append(filter_labels[fkey])
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jumlah Filter')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for fkey in ['small', 'large']:
    name = f'conv2_{fkey}_k3_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=filter_labels[fkey])
    axes[1].plot(h['val_loss'], label=filter_labels[fkey])
axes[0].set_title('Train Loss'); axes[1].set_title('Val Loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jumlah Filter')
plt.tight_layout(); plt.show()

## 7. Analisis Pengaruh Ukuran Filter

In [ ]:
kernel_labels = {'k3': '3x3', 'k5': '5x5'}
labels, scores = [], []
for kkey in ['k3', 'k5']:
    name = f'conv2_large_{kkey}_max'
    labels.append(kernel_labels[kkey])
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Ukuran Filter')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for kkey in ['k3', 'k5']:
    name = f'conv2_large_{kkey}_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=kernel_labels[kkey])
    axes[1].plot(h['val_loss'], label=kernel_labels[kkey])
axes[0].set_title('Train Loss'); axes[1].set_title('Val Loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Ukuran Filter')
plt.tight_layout(); plt.show()

## 8. Analisis Pengaruh Jenis Pooling

In [ ]:
labels, scores = [], []
for pool in ['max', 'average']:
    name = f'conv2_large_k3_{pool}'
    labels.append(pool)
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jenis Pooling')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for pool in ['max', 'average']:
    name = f'conv2_large_k3_{pool}'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=pool)
    axes[1].plot(h['val_loss'], label=pool)
axes[0].set_title('Train Loss'); axes[1].set_title('Val Loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jenis Pooling')
plt.tight_layout(); plt.show()

## 9. Rekap Semua 16 Arsitektur & Pilih Terbaik

In [ ]:
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['macro_f1'], reverse=True)

print(f"{'Nama':<35} {'Macro F1':>10}")
print('-' * 47)
for name, res in sorted_results:
    print(f"{name:<35} {res['macro_f1']:>10.4f}")

BEST_NAME   = sorted_results[0][0]
BEST_CONFIG = sorted_results[0][1]['config']
print(f'\nTerbaik: {BEST_NAME}')
print(f'Filters : {BEST_CONFIG["filters"]}')
print(f'Kernels : {BEST_CONFIG["kernel_sizes"]}')
print(f'Pooling : {BEST_CONFIG["pooling_type"]}')
print(f'Layers  : {BEST_CONFIG["num_conv_layers"]}')

In [ ]:
all_labels = [name for name, _ in sorted_results]
all_scores = [res['macro_f1'] for _, res in sorted_results]
plot_f1_comparison(all_labels, all_scores, title='Macro F1 — Semua 16 Arsitektur')

## 10. Feature Extraction

In [ ]:
train_paths = glob.glob(os.path.join(DATA_DIR, 'seg_train', 'seg_train', '**', '*.jpg'), recursive=True)
test_paths  = glob.glob(os.path.join(DATA_DIR, 'seg_test',  'seg_test',  '**', '*.jpg'), recursive=True)
print(f'Train: {len(train_paths)}, Test: {len(test_paths)}')

In [ ]:
encoder = build_cnn_model(
    BEST_CONFIG['num_conv_layers'], BEST_CONFIG['filters'],
    BEST_CONFIG['kernel_sizes'],    BEST_CONFIG['pooling_type']
)
encoder.load_weights(os.path.join(WEIGHTS_DIR, f'{BEST_NAME}.weights.h5'))
encoder.trainable = False

FEATURES_DIR = os.path.join(DRIVE_ROOT, 'features')
os.makedirs(FEATURES_DIR, exist_ok=True)

extract_features_cnn(train_paths, os.path.join(FEATURES_DIR, 'features_train.npy'), encoder)
extract_features_cnn(test_paths,  os.path.join(FEATURES_DIR, 'features_test.npy'),  encoder)

In [ ]:
features_train = np.load(os.path.join(FEATURES_DIR, 'features_train.npy'), allow_pickle=True).item()
features_test  = np.load(os.path.join(FEATURES_DIR, 'features_test.npy'),  allow_pickle=True).item()
sample_key     = list(features_train.keys())[0]
print(f'Train: {len(features_train)}, Test: {len(features_test)}')
print(f'Feature shape: {features_train[sample_key].shape}')

## 11. Perbandingan Keras vs From Scratch (Conv2D)

In [ ]:
best_model = build_cnn_model(
    BEST_CONFIG['num_conv_layers'], BEST_CONFIG['filters'],
    BEST_CONFIG['kernel_sizes'],    BEST_CONFIG['pooling_type']
)
best_model.load_weights(os.path.join(WEIGHTS_DIR, f'{BEST_NAME}.weights.h5'))
best_model.summary()

In [ ]:
n_layers  = BEST_CONFIG['num_conv_layers']
best_pool = BEST_CONFIG['pooling_type']

scratch_layers = []
layer_idx = 1

for i in range(n_layers):
    scratch_layers.append(Conv2DLayer(
        keras_layer=best_model.layers[layer_idx],
        activation='relu', strides=(1,1), padding='same'
    ))
    layer_idx += 1
    if best_pool == 'max':
        scratch_layers.append(MaxPooling2DLayer(pool_size=(2,2), strides=(2,2)))
    else:
        scratch_layers.append(AveragePooling2DLayer(pool_size=(2,2), strides=(2,2)))
    layer_idx += 1

scratch_layers.append(GlobalAveragePooling2DLayer())
layer_idx += 1
scratch_layers.append(DenseLayer(keras_layer=best_model.layers[layer_idx]))
scratch_layers.append(ActivationLayer('relu'))
layer_idx += 1
layer_idx += 1  # skip Dropout
scratch_layers.append(DenseLayer(keras_layer=best_model.layers[layer_idx]))
scratch_layers.append(ActivationLayer('softmax'))

scratch_model = CNNScratch(scratch_layers)
print(f'Jumlah layer scratch: {len(scratch_model.layers)}')

In [ ]:
N_EVAL = 200

keras_res   = evaluate_keras_model(best_model,     X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)
scratch_res = evaluate_scratch_model(scratch_model, X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)

print(f'Keras   F1 : {keras_res["macro_f1"]:.4f}')
print(f'Scratch F1 : {scratch_res["macro_f1"]:.4f}')
print(f'Selisih    : {abs(keras_res["macro_f1"] - scratch_res["macro_f1"]):.6f}')
print(f'Beda pred  : {np.sum(keras_res["y_pred"] != scratch_res["y_pred"])} dari {N_EVAL}')

In [ ]:
plot_f1_comparison(['Keras', 'From Scratch'],
                   [keras_res['macro_f1'], scratch_res['macro_f1']],
                   title='Conv2D — Keras vs From Scratch')
print('=== Keras ==='); print(keras_res['report'])
print('=== Scratch ==='); print(scratch_res['report'])

## 12. Training LocallyConnected2D
> **SKIP cell ini dan jalankan cell berikutnya jika sudah pernah ditraining**

In [ ]:
lc_model = build_locally_connected_model(
    filters=BEST_CONFIG['filters'],
    kernel_sizes=BEST_CONFIG['kernel_sizes'],
    pooling_type=BEST_CONFIG['pooling_type']
)
lc_model.summary()

lc_history = lc_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)

lc_model.save_weights(os.path.join(WEIGHTS_DIR, 'locally_connected_best.weights.h5'))

with open(os.path.join(WEIGHTS_DIR, 'lc_history.json'), 'w') as f:
    json.dump(lc_history.history, f)

print('Bobot & history LC disimpan ke Drive')

## 13. Perbandingan Shared vs Non-Shared

In [ ]:
lc_model = build_locally_connected_model(
    filters=BEST_CONFIG['filters'],
    kernel_sizes=BEST_CONFIG['kernel_sizes'],
    pooling_type=BEST_CONFIG['pooling_type']
)
lc_model.load_weights(os.path.join(WEIGHTS_DIR, 'locally_connected_best.weights.h5'))

with open(os.path.join(WEIGHTS_DIR, 'lc_history.json')) as f:
    lc_hist = json.load(f)

shared_res    = evaluate_keras_model(best_model, X_test, y_test, CLASS_NAMES)
nonshared_res = evaluate_keras_model(lc_model,   X_test, y_test, CLASS_NAMES)

print(f'Conv2D F1          : {shared_res["macro_f1"]:.4f}')
print(f'LocallyConnected F1: {nonshared_res["macro_f1"]:.4f}')
print(f'Param Conv2D       : {best_model.count_params():,}')
print(f'Param LC           : {lc_model.count_params():,}')

In [ ]:
best_hist = results[BEST_NAME]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(best_hist['loss'],    label='Conv2D')
axes[0].plot(lc_hist['loss'],      label='LocallyConnected')
axes[1].plot(best_hist['val_loss'], label='Conv2D')
axes[1].plot(lc_hist['val_loss'],   label='LocallyConnected')
axes[0].set_title('Train Loss'); axes[1].set_title('Val Loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.suptitle('Shared vs Non-Shared')
plt.tight_layout(); plt.show()

plot_f1_comparison(
    ['Conv2D\n(shared)', 'LocallyConnected\n(non-shared)'],
    [shared_res['macro_f1'], nonshared_res['macro_f1']],
    title='Shared vs Non-Shared — Macro F1'
)

## 14. From Scratch — LocallyConnected2D

In [ ]:
lc_scratch_layers = []
layer_idx = 1

for i in range(len(BEST_CONFIG['filters'])):
    lc_scratch_layers.append(LocallyConnected2DLayer(
        keras_layer=lc_model.layers[layer_idx],
        activation='relu', strides=(1,1), padding='valid',
        kernel_size=(BEST_CONFIG['kernel_sizes'][i], BEST_CONFIG['kernel_sizes'][i])
    ))
    layer_idx += 1
    if BEST_CONFIG['pooling_type'] == 'max':
        lc_scratch_layers.append(MaxPooling2DLayer(pool_size=(2,2), strides=(2,2)))
    else:
        lc_scratch_layers.append(AveragePooling2DLayer(pool_size=(2,2), strides=(2,2)))
    layer_idx += 1

lc_scratch_layers.append(FlattenLayer())
layer_idx += 1
lc_scratch_layers.append(DenseLayer(keras_layer=lc_model.layers[layer_idx]))
lc_scratch_layers.append(ActivationLayer('relu'))
layer_idx += 1
layer_idx += 1  # skip Dropout
lc_scratch_layers.append(DenseLayer(keras_layer=lc_model.layers[layer_idx]))
lc_scratch_layers.append(ActivationLayer('softmax'))

lc_scratch = CNNScratch(lc_scratch_layers)
print(f'Jumlah layer LC scratch: {len(lc_scratch.layers)}')

In [ ]:
lc_keras_res   = evaluate_keras_model(lc_model,    X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)
lc_scratch_res = evaluate_scratch_model(lc_scratch, X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)

print(f'LC Keras   F1: {lc_keras_res["macro_f1"]:.4f}')
print(f'LC Scratch F1: {lc_scratch_res["macro_f1"]:.4f}')
print(f'Selisih      : {abs(lc_keras_res["macro_f1"] - lc_scratch_res["macro_f1"]):.6f}')
print(f'Beda pred    : {np.sum(lc_keras_res["y_pred"] != lc_scratch_res["y_pred"])} dari {N_EVAL}')

## 15. Rekap Akhir

In [ ]:
print('='*55)
print(f"{'Model':<40} {'Macro F1':>10}")
print('='*55)
print(f"{'Conv2D (Keras)':<40} {shared_res['macro_f1']:>10.4f}")
print(f"{'Conv2D (Scratch)':<40} {scratch_res['macro_f1']:>10.4f}")
print(f"{'LocallyConnected (Keras)':<40} {nonshared_res['macro_f1']:>10.4f}")
print(f"{'LocallyConnected (Scratch)':<40} {lc_scratch_res['macro_f1']:>10.4f}")
print('='*55)
print(f'Param Conv2D : {best_model.count_params():,}')
print(f'Param LC     : {lc_model.count_params():,}')

In [ ]:
plot_f1_comparison(
    ['Conv2D\nKeras', 'Conv2D\nScratch', 'LC\nKeras', 'LC\nScratch'],
    [shared_res['macro_f1'], scratch_res['macro_f1'],
     nonshared_res['macro_f1'], lc_scratch_res['macro_f1']],
    title='Rekap Akhir — Macro F1'
)

## 16. BONUS

### Bonus 1 — Visualisasi Feature Maps + Grad-CAM

In [ ]:
!pip install opencv-python-headless -q
import matplotlib.cm as cm
import cv2

def get_feature_maps(model, img, layer_names=None):
    if layer_names is None:
        layer_names = [l.name for l in model.layers if 'conv2d' in l.name]
    feature_model = keras.Model(
        inputs  = model.input,
        outputs = [model.get_layer(n).output for n in layer_names]
    )
    outputs = feature_model.predict(img[np.newaxis], verbose=0)
    if len(layer_names) == 1:
        outputs = [outputs]
    return layer_names, outputs


def visualize_feature_maps(model, img, n_filters=8):
    layer_names, outputs = get_feature_maps(model, img)
    for name, fmap in zip(layer_names, outputs):
        fmap   = fmap[0]
        n_show = min(n_filters, fmap.shape[-1])
        fig, axes = plt.subplots(1, n_show, figsize=(2*n_show, 2.5))
        fig.suptitle(f'Feature Maps — {name}')
        for i, ax in enumerate(axes):
            ch = fmap[:, :, i]
            ch = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
            ax.imshow(ch, cmap='viridis')
            ax.set_title(f'Filter {i}', fontsize=8)
            ax.axis('off')
        plt.tight_layout(); plt.show()

In [ ]:
def grad_cam(model, img, class_idx=None, last_conv_layer_name=None):
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if 'conv2d' in layer.name:
                last_conv_layer_name = layer.name
                break

    grad_model = keras.Model(
        inputs  = model.input,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img[np.newaxis].astype(np.float32))
        if class_idx is None:
            class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads    = tape.gradient(loss, conv_outputs)[0].numpy()
    conv_out = conv_outputs[0].numpy()
    weights  = np.mean(grads, axis=(0, 1))

    cam = np.zeros(conv_out.shape[:2], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * conv_out[:, :, i]

    cam = np.maximum(cam, 0)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = cv2.resize(cam, (img.shape[1], img.shape[0]))
    return cam, int(class_idx)


def overlay_grad_cam(img, cam, alpha=0.4):
    heatmap = cm.jet(np.uint8(255 * cam))[:, :, :3]
    return np.clip(heatmap * alpha + img * (1 - alpha), 0, 1)


def visualize_grad_cam(model, images, y_true, class_names, n_samples=3):
    y_pred_proba = model.predict(images, verbose=0)
    confidence   = np.max(y_pred_proba, axis=1)
    sorted_idx   = np.argsort(confidence)[::-1]

    high_idx = sorted_idx[:n_samples]
    mid_idx  = sorted_idx[len(sorted_idx)//2 : len(sorted_idx)//2 + n_samples]
    low_idx  = sorted_idx[-n_samples:]

    for label, idxs in [('High', high_idx), ('Mid', mid_idx), ('Low', low_idx)]:
        fig, axes = plt.subplots(n_samples, 3, figsize=(12, 3*n_samples))
        fig.suptitle(f'Grad-CAM — {label} Confidence')
        for row, idx in enumerate(idxs):
            img       = images[idx]
            cam, pred = grad_cam(model, img)
            overlay   = overlay_grad_cam(img, cam)
            axes[row, 0].imshow(img);          axes[row, 0].set_title(f'True: {class_names[y_true[idx]]}'); axes[row, 0].axis('off')
            axes[row, 1].imshow(cam, cmap='jet'); axes[row, 1].set_title('Grad-CAM');                       axes[row, 1].axis('off')
            axes[row, 2].imshow(overlay);      axes[row, 2].set_title(f'Pred: {class_names[pred]} ({confidence[idx]:.2f})'); axes[row, 2].axis('off')
        plt.tight_layout(); plt.show()

In [ ]:
print(f'Feature maps untuk: {CLASS_NAMES[y_test[0]]}')
visualize_feature_maps(best_model, X_test[0], n_filters=8)

In [ ]:
visualize_grad_cam(best_model, X_test[:50], y_test[:50], CLASS_NAMES, n_samples=3)

### Bonus 2 — Batch Inference

In [ ]:
for bs in [1, 8, 32]:
    preds = scratch_model.predict_batch(X_test[:32], batch_size=bs)
    print(f'batch_size={bs:2d} → shape: {preds.shape}, sample: {preds[:5]}')

pred_batch  = scratch_model.predict_batch(X_test[:10], batch_size=10)
pred_single = np.array([scratch_model.predict(X_test[i]) for i in range(10)])
print(f'\nBatch  : {pred_batch}')
print(f'Single : {pred_single}')
print(f'Konsisten: {np.all(pred_batch == pred_single)}')

### Bonus 3 — Backward Propagation

In [ ]:
test_img    = X_test[0:2]
kernel_test = np.random.randn(3, 3, 3, 8).astype(np.float32) * 0.1
bias_test   = np.zeros(8, dtype=np.float32)
conv_test   = Conv2DLayer(kernel=kernel_test, bias=bias_test, activation='relu', padding='same')

out    = conv_test.forward(test_img)
dx     = conv_test.backward(np.ones_like(out))
print(f'Conv2D   : {test_img.shape} → {out.shape} → dx {dx.shape} ✓')

pool_test = MaxPooling2DLayer(pool_size=(2,2))
out_pool  = pool_test.forward(test_img)
dx_pool   = pool_test.backward(np.ones_like(out_pool))
print(f'MaxPool  : {test_img.shape} → {out_pool.shape} → dx {dx_pool.shape} ✓')

gap_test = GlobalAveragePooling2DLayer()
out_gap  = gap_test.forward(test_img)
dx_gap   = gap_test.backward(np.ones_like(out_gap))
print(f'GAP      : {test_img.shape} → {out_gap.shape} → dx {dx_gap.shape} ✓')

flat_test = FlattenLayer()
out_flat  = flat_test.forward(test_img)
dx_flat   = flat_test.backward(np.ones_like(out_flat))
print(f'Flatten  : {test_img.shape} → {out_flat.shape} → dx {dx_flat.shape} ✓')

act_test = ActivationLayer('relu')
x_act    = np.random.randn(2, 10).astype(np.float32)
out_act  = act_test.forward(x_act)
dx_act   = act_test.backward(np.ones_like(out_act))
print(f'ReLU     : {x_act.shape} → {out_act.shape} → dx {dx_act.shape} ✓')